# Import Data from Yahoo Finance

In [ ]:
!pip install -q yfinance

import yfinance as yf
import pandas as pd
import numpy as np

In [ ]:
df = yf.download(
    "SPY",
    start="1993-01-29",
    end="2026-11-20",  # end date is exclusive, so add 1 day to include Nov 16
    interval="1d",
    auto_adjust=False,
    multi_level_index=False
)

# Preview the latest rows in descending order, matching your screenshot
df.sort_index(ascending=False).head(10)

[*********************100%***********************]  1 of 1 completed


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2026-11-19,715.869995,715.869995,739.900024,715.650024,737.460022,76476400
2026-11-18,730.200012,730.200012,735.059998,726.799988,733.369995,51804800
2026-11-17,732.440002,732.440002,739.760010,727.000000,738.390015,71636300
2026-11-16,739.950012,739.950012,749.820007,738.070007,749.659973,69207300
2026-11-13,754.369995,754.369995,760.119995,747.880005,760.070007,38720000
2026-11-12,757.119995,757.119995,763.039978,753.669983,753.979980,37460600
2026-11-11,759.809998,759.809998,762.510010,754.900024,756.799988,39151000
2026-11-10,749.919983,749.919983,752.140015,744.859985,747.940002,40698600
2026-11-09,752.690002,752.690002,755.419983,741.030029,741.950012,50281200


# Add features

In [ ]:
df.index = pd.to_datetime(df.index)
df = df.sort_index()

eps = 1e-8
c_prev = df["Close"].shift(1)
day_range = df["High"] - df["Low"] + eps

# 1. Multi-Horizon Log Returns & Gaps
for k in [1, 3, 5, 10]:
    df[f"MHLR_{k}"] = np.log(df["Close"] / df["Close"].shift(k))

df["Overnight_Gap"] = (df["Open"] - c_prev) / c_prev
df["Intraday_Ret"] = (df["Close"] - df["Open"]) / df["Open"]

# 2. Candlestick Geometry (for Limit order fills & reversals)
df["(High - Low)/Close"] = (df["High"] - df["Low"]) / df["Close"]
df["IBS"] = (df["Close"] - df["Low"]) / day_range
df["Lower_Wick"] = (np.minimum(df["Open"], df["Close"]) - df["Low"]) / day_range
df["Upper_Wick"] = (df["High"] - np.maximum(df["Open"], df["Close"])) / day_range
df["Body_Ratio"] = np.abs(df["Close"] - df["Open"]) / day_range

# 3. Volatility Estimators (for Put timing & Stop/Limit offsets)
true_range = np.maximum(
    df["High"] - df["Low"],
    np.maximum((df["High"] - c_prev).abs(), (df["Low"] - c_prev).abs())
)
atr_5 = true_range.ewm(alpha=1/5, adjust=False).mean()
atr_14 = true_range.ewm(alpha=1/14, adjust=False).mean()
atr_20 = true_range.ewm(alpha=1/20, adjust=False).mean()

df["NATR_14"] = atr_14 / df["Close"]
df["Vol_Regime"] = atr_5 / (atr_20 + eps)
df["Parkinson_10"] = np.sqrt((np.log(df["High"] / df["Low"]) ** 2).rolling(10).mean() / (4 * np.log(2)))

# 4. Trend & Mean-Reversion Indicators
for n in [10, 50, 200]:
    df[f"Dist_SMA_{n}"] = (df["Close"] / df["Close"].rolling(n).mean()) - 1.0

sma_20 = df["Close"].rolling(20).mean()
std_20 = df["Close"].rolling(20).std()
df["BB_Pct_B"] = (df["Close"] - (sma_20 - 2 * std_20)) / (4 * std_20 + eps)

delta = df["Close"].diff()
gain = delta.clip(lower=0).ewm(alpha=1/14, adjust=False).mean()
loss = (-delta.clip(upper=0)).ewm(alpha=1/14, adjust=False).mean()
df["RSI_14"] = 1.0 - (1.0 / (1.0 + (gain / (loss + eps))))  # Normalized [0, 1]

low_20 = df["Low"].rolling(20).min()
high_20 = df["High"].rolling(20).max()
df["Donchian_20"] = (df["Close"] - low_20) / (high_20 - low_20 + eps)

# 5. Volume Dynamics
df["Rel_Volume_20"] = df["Volume"] / (df["Volume"].rolling(20).mean() + eps)
df["Vol_Price_Force"] = df["MHLR_1"] * df["Rel_Volume_20"]

# 6. Cyclical Calendar Features (from Date index)
dow = df.index.dayofweek
dom = df.index.day
month = df.index.month

df["DOW_sin"] = np.sin(2 * np.pi * dow / 5.0)
df["DOW_cos"] = np.cos(2 * np.pi * dow / 5.0)
df["DOM_sin"] = np.sin(2 * np.pi * dom / 31.0)
df["DOM_cos"] = np.cos(2 * np.pi * dom / 31.0)
df["Month_sin"] = np.sin(2 * np.pi * month / 12.0)
df["Month_cos"] = np.cos(2 * np.pi * month / 12.0)

# 7. Clean up warm-up NaNs and isolate stationary features for the MLP
df = df.dropna()

raw_cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume", "Close - High", "Close - Low", "MHLR"]
feature_cols = [c for c in df.columns if c not in raw_cols]
df[feature_cols].head(10)

,MHLR_1,MHLR_3,MHLR_5,MHLR_10,Overnight_Gap,Intraday_Ret,(High - Low)/Close,IBS,Lower_Wick,Upper_Wick,...,RSI_14,Donchian_20,Rel_Volume_20,Vol_Price_Force,DOW_sin,DOW_cos,DOM_sin,DOM_cos,Month_sin,Month_cos
Date,,,,,,,,,,,,,,,,,,,,,
1993-11-11,-0.002692,0.005405,0.011522,-0.010057,0.000000,-0.002688,0.006065,0.111111,0.111111,0.444444,...,0.499982,0.519231,0.276396,-0.000744,-0.587785,-0.809017,0.790776,-0.612106,-0.5,0.866025
1993-11-12,0.004706,0.009434,0.011467,-0.005351,0.002022,0.002690,0.006707,0.500000,0.100000,0.500000,...,0.541835,0.708333,0.429493,0.002021,-0.951057,0.309017,0.651372,-0.758758,-0.5,0.866025
1993-11-15,-0.000671,0.001343,0.009440,-0.008687,0.002012,-0.002677,0.005369,0.500000,0.500000,0.000000,...,0.534947,0.702128,1.067246,-0.000716,0.000000,1.000000,0.101168,-0.994869,-0.5,0.866025
1993-11-16,0.004687,0.008722,0.013450,-0.003334,0.002013,0.002679,0.007348,0.909091,0.545455,0.090909,...,0.575620,0.851064,2.361514,0.011068,0.951057,0.309017,-0.101168,-0.994869,-0.5,0.866025
1993-11-17,-0.005358,-0.001342,0.000672,0.004038,0.000668,-0.006008,0.008731,0.307692,0.307692,0.000000,...,0.519680,0.680851,0.198940,-0.001066,0.587785,-0.809017,-0.299363,-0.954139,-0.5,0.866025
1993-11-18,-0.002690,-0.003361,0.000674,0.012195,-0.001343,-0.001345,0.006061,0.444444,0.444444,0.333333,...,0.493838,0.595745,1.156108,-0.003110,-0.587785,-0.809017,-0.485302,-0.874347,-0.5,0.866025
1993-11-19,-0.002022,-0.010071,-0.006055,0.005413,-0.003367,0.001351,0.003374,0.600000,0.200000,0.400000,...,0.474770,0.531915,0.546251,-0.001105,-0.951057,0.309017,-0.651372,-0.758758,-0.5,0.866025
1993-11-22,-0.006091,-0.010804,-0.011475,-0.002035,-0.002699,-0.003383,0.007468,0.454545,0.454545,0.090909,...,0.422111,0.340426,0.840836,-0.005122,0.000000,1.000000,-0.968077,-0.250653,-0.5,0.866025
1993-11-23,0.005416,-0.002697,-0.010746,0.002705,0.005431,0.000000,0.006077,0.888889,0.888889,0.111111,...,0.477579,0.510638,0.467760,0.002534,0.951057,0.309017,-0.998717,-0.050649,-0.5,0.866025


In [ ]:
# target classes
HORIZON = 3
THRESHOLD = 0.005
future_ret = df["Close"].shift(-HORIZON) / df["Close"] - 1.0
df["Target"] = 1
df.loc[future_ret > THRESHOLD, "Target"] = 2
df.loc[future_ret < -THRESHOLD, "Target"] = 0
df = df.iloc[:-HORIZON].copy()
n = len(df)

# train test validate split
train_end = int(0.8*n)
val_end = int(0.1*n)
test_end = n

train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:train_end+val_end]
test_df = df.iloc[train_end+val_end:test_end]

In [ ]:
# scaler
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
X_train_2d = scaler.fit_transform(train_df[feature_cols].to_numpy(dtype=np.float32))
X_val_2d   = scaler.transform(val_df[feature_cols].to_numpy(dtype=np.float32))
X_test_2d  = scaler.transform(test_df[feature_cols].to_numpy(dtype=np.float32))

WINDOW_SIZE = 10
def build_xy(X_2d, y_1d, w_size=WINDOW_SIZE):
    w = sliding_window_view(X_2d, window_shape=w_size, axis=0)
    return np.transpose(w, (0, 2, 1)).reshape(w.shape[0], -1), y_1d[w_size - 1:]

X_train, y_train = build_xy(X_train_2d, train_df["Target"].to_numpy(dtype=np.int64))
X_val, y_val     = build_xy(X_val_2d,   val_df["Target"].to_numpy(dtype=np.int64))
X_test, y_test   = build_xy(X_test_2d,  test_df["Target"].to_numpy(dtype=np.int64))


In [ ]:
# mlp def
import torch
import torch.nn as nn
class MLP(nn.Module):
  def __init__(self, input_dim: int, num_classes: int = 3, dropout: float = 0.3):
    super(MLP, self).__init__()
    self.net = nn.Sequential(
        # HiddenLayer 1
        nn.Linear(input_dim, 128),
        nn.BatchNorm1d(128), #mean of 0, deviation of 1 to avoid 2008 and 2021 fucking everything up
        nn.ReLU(),
        nn.Dropout(dropout),

        #Hidden Layer 2
        nn.Linear(128, 64),
        nn.BatchNorm1d(64),
        nn.ReLU(),
        nn.Dropout(dropout),

        # HiddenLayer 3
        nn.Linear(64, 32),
        nn.BatchNorm1d(32),
        nn.ReLU(),

        # Output Layer
        nn.Linear(32, num_classes)
    )
  def forward(self, x : torch.Tensor) -> torch.Tensor:
    return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MLP(input_dim=X_train.shape[1], num_classes=3, dropout=0.3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
# train model
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.utils.class_weight import compute_class_weight

BATCH_SIZE = 64

train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long)),
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_loader = DataLoader(
    TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)),
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1, 2]), y=y_train)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

EPOCHS = 50
PATIENCE = 10
best_val_loss = float("inf")
best_weights = copy.deepcopy(model.state_dict())
epochs_no_improve = 0

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)
        train_correct += (logits.argmax(dim=1) == y_batch).sum().item()
        train_total += y_batch.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            val_loss += loss.item() * X_batch.size(0)
            val_correct += (logits.argmax(dim=1) == y_batch).sum().item()
            val_total += y_batch.size(0)

    train_loss /= train_total
    val_loss /= val_total
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} (Acc: {train_correct/train_total:.3f}) | Val Loss: {val_loss:.4f} (Acc: {val_correct/val_total:.3f})")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (Best Val Loss: {best_val_loss:.4f})")
            break

model.load_state_dict(best_weights)

# Test Evaluation
model.eval()
all_probs, all_preds, all_targets = [], [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        probs = torch.softmax(model(X_batch.to(device)), dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.append(probs.argmax(axis=1))
        all_targets.append(y_batch.numpy())

probs = np.vstack(all_probs)
print("\nConfusion Matrix:\n", confusion_matrix(np.concatenate(all_targets), np.concatenate(all_preds)))
print("\nClassification Report:\n", classification_report(np.concatenate(all_targets), np.concatenate(all_preds), target_names=["Sell (0)", "Hold (1)", "Buy (2)"]))

Epoch 01 | Train Loss: 1.0869 (Acc: 0.378) | Val Loss: 1.0884 (Acc: 0.382)
Epoch 02 | Train Loss: 1.0573 (Acc: 0.417) | Val Loss: 1.0881 (Acc: 0.351)
Epoch 03 | Train Loss: 1.0512 (Acc: 0.425) | Val Loss: 1.0795 (Acc: 0.376)
Epoch 04 | Train Loss: 1.0443 (Acc: 0.437) | Val Loss: 1.0781 (Acc: 0.384)
Epoch 05 | Train Loss: 1.0406 (Acc: 0.447) | Val Loss: 1.0827 (Acc: 0.398)
Epoch 06 | Train Loss: 1.0344 (Acc: 0.447) | Val Loss: 1.0773 (Acc: 0.412)
Epoch 07 | Train Loss: 1.0334 (Acc: 0.436) | Val Loss: 1.0862 (Acc: 0.368)
Epoch 08 | Train Loss: 1.0267 (Acc: 0.457) | Val Loss: 1.0826 (Acc: 0.406)
Epoch 09 | Train Loss: 1.0215 (Acc: 0.455) | Val Loss: 1.0859 (Acc: 0.403)
Epoch 10 | Train Loss: 1.0131 (Acc: 0.466) | Val Loss: 1.0905 (Acc: 0.395)
Epoch 11 | Train Loss: 1.0063 (Acc: 0.481) | Val Loss: 1.0976 (Acc: 0.369)
Epoch 12 | Train Loss: 1.0098 (Acc: 0.472) | Val Loss: 1.0939 (Acc: 0.400)
Epoch 13 | Train Loss: 0.9983 (Acc: 0.479) | Val Loss: 1.0937 (Acc: 0.401)
Epoch 14 | Train Loss: 1.

In [ ]:
df["MHLR_1"].head(64)

,MHLR_1
Date,
1993-11-11,-0.002692
1993-11-12,0.004706
1993-11-15,-0.000671
1993-11-16,0.004687
1993-11-17,-0.005358
...,...
1994-02-04,-0.023020
1994-02-07,0.004647
1994-02-08,0.000662
